# TinyBERT-L2 Reranker

**Model:** cross-encoder/ms-marco-TinyBERT-L2-v2 (Apache-2.0) | **Weights:** 17.6MB, 4.4M parameters | **Product:** prod-4llv34zbqxmtm

Re-ranks candidate passages for a query so the best ones come first in a RAG or search pipeline. It scores
passages you already retrieved; it does not search your documents itself.

Measured on 1,000 MS MARCO v1.1 validation queries (about 8.3 candidates each): MRR@10 0.583 vs 0.352 for random
order. Evaluate on your own data before relying on it.

## Cost
Software fee on the listing plus the SageMaker instance. Recommended real-time instance: **ml.t2.medium**
(burstable; choose a larger type for sustained heavy traffic). Batch transform: ml.m5.large.
Delete the endpoint when you are done (last cell).

In [ ]:
import json
import boto3
import sagemaker
from sagemaker import ModelPackage

region = boto3.Session().region_name
role = sagemaker.get_execution_role()
sm_client = boto3.client('sagemaker', region_name=region)
print(f'Region: {region}')
print(f'Role: {role}')

In [ ]:
# Replace with the Model Package ARN shown after you subscribe on AWS Marketplace
model_package_arn = 'arn:aws:sagemaker:REGION:ACCOUNT:model-package/MODEL_PACKAGE_NAME'

if 'REGION' in model_package_arn or 'ACCOUNT' in model_package_arn or 'MODEL_PACKAGE_NAME' in model_package_arn:
    raise ValueError('Set model_package_arn to the ARN of your Marketplace subscription before deploying.')

endpoint_name = 'tinybert-l2-reranker'
instance_type = 'ml.t2.medium'

model = ModelPackage(role=role, model_package_arn=model_package_arn, sagemaker_session=sagemaker.Session())
predictor = model.deploy(initial_instance_count=1, instance_type=instance_type, endpoint_name=endpoint_name)
print(f'Endpoint deployed: {endpoint_name}')

## Step 2: Rank passages

Send one query and up to 512 candidate passages. The response has raw `scores` (higher = more relevant, compare only
within one query), 0-1 `relevance` values, `ranking` (passage indices, best first) and an optional `top_k` list.

In [ ]:
runtime = boto3.client('sagemaker-runtime', region_name=region)

query = 'how do I reset my password'
passages = [
    'Our office is closed on public holidays.',
    'To reset your password, open Settings, choose Security and select Reset password.',
    'Passwords must contain at least twelve characters.',
    'The marathon route passes through the city centre.',
]

response = runtime.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType='application/json',
    Body=json.dumps({'query': query, 'passages': passages, 'top_k': 2}),
)
result = json.loads(response['Body'].read().decode('utf-8'))

for rank, idx in enumerate(result['ranking'], start=1):
    print(f"{rank}. score={result['scores'][idx]:.2f} relevance={result['relevance'][idx]:.4f}  {passages[idx]}")

## Step 3: Batch transform (optional)

For offline jobs, put one JSON request per file (same format as above) under an S3 prefix and run a transform job
with `ContentType='application/json'`. Each input file is sent as a single request.

In [ ]:
# Cleanup - delete the endpoint (and its config and model) to stop charges
try:
    predictor.delete_endpoint(delete_endpoint_config=True)
    predictor.delete_model()
    print(f'Endpoint {endpoint_name} deleted.')
except Exception as e:
    print(f'Cleanup failed: {e}')